# Lesson 16 — chunking · หั่นข้อความยาวก่อนเก็บ

โพสต์ยาวหนึ่งโพสต์ embed เป็น vector เดียว ความหมายเฉลี่ยกันจนจาง
หั่นเป็นชิ้นเล็ก แต่ละชิ้นมี vector ของตัวเอง ค้นเจอชิ้น แล้วค่อยเอาโพสต์แม่มาแสดง
บทนี้ไม่มี API ใหม่ของ LanceDB เลย เป็นเรื่องออกแบบ schema ล้วน ๆ

ภาษาไทยไม่มีช่องว่างระหว่างคำ หั่นตามจำนวนตัวอักษรคือวิธีตรงไปตรงมาที่สุด
40 ตัวอักษร ทับซ้อน 10 ตัว เพื่อไม่ให้คำขาดกลางแล้วหายไปทั้งสองฝั่ง
ตัดตามคำด้วย icu (บทที่ 8) ดีกว่า แต่บทนี้ขอแบบเห็นตัวเลขชัด ๆ ก่อน

In [1]:
%pip install -q lancedb pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, urllib.request, pathlib
if not pathlib.Path("../data/lesson_data.py").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Soul-Brews-Studio/lancedb-oracle/main/lessons/data/lesson_data.py", "lesson_data.py")
sys.path.insert(0, "../data")
from lesson_data import load

posts = load("nat_posts.jsonl")
longest = sorted(posts, key=lambda p: -len(p["text"]))[:3]
for p in longest:
    print(p["id"], len(p["text"]), "chars")

p01 184 chars
p08 177 chars
p07 159 chars


**หั่น** — เลื่อนหน้าต่าง 40 ตัวอักษร ก้าวทีละ 30 (40 − 10 ทับซ้อน)
แต่ละชิ้นจำว่ามาจากโพสต์ไหน (`parent_id`) ชิ้นที่เท่าไหร่ (`chunk_index`)
พิมพ์ให้ดูว่าท้ายชิ้นก่อนกับหัวชิ้นถัดไป คือตัวอักษรชุดเดียวกัน

In [3]:
SIZE, OVERLAP = 40, 10

def chunk(text, size=SIZE, overlap=OVERLAP):
    step = size - overlap
    return [text[i:i + size] for i in range(0, max(len(text) - overlap, 1), step)]

p = longest[0]
for i, c in enumerate(chunk(p["text"])):
    print(f"{p['id']}#{i}  …{c[:10]}|{c[10:-10]}|{c[-10:]}…")

p01#0  …ยอมกลับมาท|ำ Memory เพราะซาบซึ้|งว่ามันต้อ…
p01#1  …งว่ามันต้อ|งทำแล้วครับ ARRA Ora|cle Memory…
p01#2  …cle Memory| สามารถทำให้ claude |บนเว็ป คุย…
p01#3  …บนเว็ป คุย|กับชาวแก๊งอื่นๆได้แล|้วครับ แล้…
p01#4  …้วครับ แล้|วจะพาทำ Workshop สร้|าง Memory …
p01#5  …าง Memory |และ MCP ด้วยตั|วเองนะครับ…


**เก็บลงตาราง** — vector ของแต่ละชิ้นทำมือเหมือนบทก่อน ๆ
ชิ้นของโพสต์ไหน ก็ยืม vector ของโพสต์นั้น แล้วขยับนิดหน่อยตามลำดับชิ้น
(ของจริงใช้โมเดล embed ทีละชิ้น บทที่ 7)

In [4]:
import lancedb

rows = []
for p in longest:
    for i, c in enumerate(chunk(p["text"])):
        v = [round(x + 0.02 * i, 3) for x in p["vector"]]
        rows.append({"chunk_id": f"{p['id']}#{i}", "parent_id": p["id"], "chunk_index": i, "text": c, "vector": v})

db = lancedb.connect("./data")
chunks = db.create_table("chunks", data=rows, mode="overwrite")
parents = db.create_table("posts", data=longest, mode="overwrite")
print(len(rows), "chunks from", len(longest), "posts")
chunks.to_pandas()[["chunk_id", "parent_id", "chunk_index", "text"]].head(6)

17 chunks from 3 posts


[2026-09-10T11:37:10Z WARN  lance::dataset::write::insert] No existing dataset at /opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/16-chunking/data/chunks.lance, it will be created
[2026-09-10T11:37:10Z WARN  lance::dataset::write::insert] No existing dataset at /opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/16-chunking/data/posts.lance, it will be created


,chunk_id,parent_id,chunk_index,text
0,p01#0,p01,0,ยอมกลับมาทำ Memory เพราะซาบซึ้งว่ามันต้อ
1,p01#1,p01,1,งว่ามันต้องทำแล้วครับ ARRA Oracle Memory
2,p01#2,p01,2,cle Memory สามารถทำให้ claude บนเว็ป คุย
3,p01#3,p01,3,บนเว็ป คุยกับชาวแก๊งอื่นๆได้แล้วครับ แล้
4,p01#4,p01,4,้วครับ แล้วจะพาทำ Workshop สร้าง Memory
5,p01#5,p01,5,าง Memory และ MCP ด้วยตัวเองนะครับ


**ค้นชิ้น** — คำถามทิศ memory `[1, 0, 0]` ขอ 8 ชิ้น
6 ชิ้นแรกมาจาก p01 ทั้งหมด ถ้าส่งแบบนี้ให้ agent มันจะเห็นโพสต์เดิมซ้ำ 6 ครั้ง

In [5]:
hits = chunks.search([1.0, 0.0, 0.0]).limit(8).to_pandas()
hits[["chunk_id", "parent_id", "_distance", "text"]].assign(text=lambda d: d.text.str[:30])

,chunk_id,parent_id,_distance,text
0,p01#0,p01,0.0200,ยอมกลับมาทำ Memory เพราะซาบซึ้
1,p01#1,p01,0.0212,งว่ามันต้องทำแล้วครับ ARRA Ora
2,p01#2,p01,0.0248,cle Memory สามารถทำให้ claude
3,p01#3,p01,0.0308,บนเว็ป คุยกับชาวแก๊งอื่นๆได้แล
4,p01#4,p01,0.0392,้วครับ แล้วจะพาทำ Workshop สร้
5,p01#5,p01,0.0500,าง Memory และ MCP ด้วยตัวเองนะ
6,p08#0,p08,1.6200,Multi Agent แบบ Team .... ของ
7,p08#1,p08,1.6212,Claude Code เพื่อเรียก SendMes


**ยุบกลับเป็นโพสต์แม่** — group ด้วย `parent_id` เอาชิ้นที่ใกล้ที่สุดเป็นคะแนนของโพสต์
แล้ว join กลับไปเอาข้อความเต็ม
นี่คือ pattern "chunk to retrieve, parent to show" ที่ระบบ RAG ทุกตัวใช้

In [6]:
import duckdb
best = hits.groupby("parent_id", as_index=False).agg(best_distance=("_distance", "min"), best_chunk=("chunk_id", "first"))
parent_df = parents.to_pandas()[["id", "text"]]
duckdb.sql("""
    SELECT b.parent_id, b.best_distance, b.best_chunk, substr(p.text, 1, 50) AS parent_text
    FROM best b JOIN parent_df p ON p.id = b.parent_id
    ORDER BY b.best_distance
""").df()

,parent_id,best_distance,best_chunk,parent_text
0,p01,0.02,p01#0,ยอมกลับมาทำ Memory เพราะซาบซึ้งว่ามันต้องทำแล้...
1,p08,1.62,p08#0,Multi Agent แบบ Team .... ของ Claude Code เพื่...


บน disk สองตาราง `chunks` แถวเยอะกว่า `posts` แต่ข้อความรวมยาวกว่าเพราะทับซ้อน
ทับซ้อน 10/40 = จ่ายเพิ่มราว 25% (วัดจริง 27% เพราะชิ้นท้ายของแต่ละโพสต์สั้นไม่เต็ม 40) เพื่อไม่ให้คำขาด

In [7]:
from pathlib import Path
total_chunk_chars = sum(len(r["text"]) for r in rows)
total_post_chars = sum(len(p["text"]) for p in longest)
print(f"post chars  {total_post_chars}")
print(f"chunk chars {total_chunk_chars}  (+{100 * (total_chunk_chars / total_post_chars - 1):.0f}%)")
for name in ("posts", "chunks"):
    n = sum(f.stat().st_size for f in Path(f"data/{name}.lance/data").glob("*.lance"))
    print(f"{name:<7} {n:>6} bytes on disk")

post chars  520
chunk chars 660  (+27%)
posts     2628 bytes on disk
chunks    3602 bytes on disk
